In [1]:
library(data.table)
library(yaml)

BASE      <- path.expand("~/resultados_pandemia")
STEP      <- 25
MAX_ENVIO <- 20000

dirs <- list.dirs(BASE, recursive = FALSE, full.names = TRUE)
dirs <- dirs[grepl("WF2026080", basename(dirs))]

tb <- rbindlist(lapply(dirs, function(d) {
  f <- file.path(d, "PARAM.yml")
  if (!file.exists(f)) return(NULL)
  p <- yaml::read_yaml(f)
  data.table(
    celda            = p$celda,
    semilla          = p$semilla_primigenia,
    n_excluidos      = length(p$meses_excluidos),
    meses_excluidos  = paste(as.integer(p$meses_excluidos), collapse = " "),
    n_train          = length(p$trainingstrategy$training),
    n_final_train    = length(p$trainingstrategy$final_train),
    auc              = p$out$lgbm$AUC,
    num_iterations   = p$out$lgbm$mejores_hiperparametros$num_iterations,
    num_leaves       = p$out$lgbm$mejores_hiperparametros$num_leaves,
    min_data_in_leaf = p$out$lgbm$mejores_hiperparametros$min_data_in_leaf,
    feature_fraction = p$out$lgbm$mejores_hiperparametros$feature_fraction,
    ganancia         = p$resultado$ganancia_suavizada_max,
    envios           = p$resultado$envios
  )
}), fill = TRUE)

setorder(tb, celda, semilla)

curvas <- rbindlist(lapply(dirs, function(d) {
  f <- file.path(d, "ganancias.txt")
  if (!file.exists(f)) return(NULL)
  p <- yaml::read_yaml(file.path(d, "PARAM.yml"))
  g <- fread(f, select = c("gan_acum", "gan_suavizada"))
  g[, envio := .I]
  g <- g[envio <= MAX_ENVIO][envio %% STEP == 0]
  g[, `:=`(celda = p$celda, semilla = p$semilla_primigenia)]
  g[]
}), fill = TRUE)

cat("\n--- corridas por celda ---\n");    print(tb[, .N, by = celda])
cat("\n--- exclusiones por celda ---\n")
print(unique(tb[, .(celda, meses_excluidos, n_train, n_final_train)]))
cat("\n--- resumen ---\n")
print(tb[, .(n = .N,
             gan_media = round(mean(ganancia), 3),
             gan_sd    = round(sd(ganancia), 3),
             iters_med = median(num_iterations),
             auc_media = round(mean(auc), 5)), by = celda])
cat("\n--- curvas cargadas ---\n");       print(curvas[, .N, by = celda])

fwrite(tb,     "~/consolidado_resumen.csv")
fwrite(curvas, "~/consolidado_curvas.csv")


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%





--- corridas por celda ---
    celda     N
   <char> <int>
1:      A    10
2:      B    10
3:      C    10
4:      D    10

--- exclusiones por celda ---
    celda                                                       meses_excluidos
   <char>                                                                <char>
1:      A                                                                      
2:      B                                                         202003 202004
3:      C                                                                202006
4:      D 202003 202004 202005 202006 202007 202008 202009 202010 202011 202012
   n_train n_final_train
     <int>         <int>
1:      27            29
2:      25            27
3:      26            28
4:      17            19

--- resumen ---
    celda     n gan_media gan_sd iters_med auc_media
   <char> <int>     <num>  <num>     <num>     <num>
1:      A    10    92.869  2.625     288.0   0.93231
2:      B    10    93.682  1.880     31